# Backend Benchmark Exploration

Use this notebook first. It runs the ChromaDB vs pgvector benchmark and shows pandas DataFrames for speed, retrieval quality, scores, and ranking parity. Nothing is written to `evaluation_results.json` unless you run the optional save cell at the end.

## Before Running

- Make sure `.env` points `PDF_SOURCE_DIR` at your PDFs.
- Make sure chunks already exist under `data/interim/chunks/`; if not, run `python pipeline.py extract` first.
- For pgvector, run `docker compose up -d`, `alembic upgrade head`, and `python scripts/check_pgvector.py`.
- If imports fail, check the kernel path in the next cell. VS Code/Jupyter must use the same environment where the project dependencies are installed.

In [1]:
import sys

print(sys.executable)
print(sys.version)
# If this path is not your project environment, switch kernels in VS Code.
# To install dependencies into this exact kernel, run:
# %pip install -r requirements.txt

/opt/anaconda3/envs/project-d/bin/python
3.12.13 | packaged by Anaconda, Inc. | (main, Mar 19 2026, 20:12:32) [Clang 20.1.8 ]


In [2]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "pipeline.py").exists():
    REPO_ROOT = REPO_ROOT.parent

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from benchmark_metrics import (
    run_backend_benchmark,
    summarise_results,
    write_results_json,
)

pd.set_option("display.max_colwidth", 140)
pd.set_option("display.precision", 6)

/opt/anaconda3/envs/project-d/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Parameters

For a quick smoke test, set `MAX_CHUNKS` or `MAX_QUERIES` to a small number. For the professor's spec, eventually use 20 representative queries and `QUERY_REPEATS = 3`.

In [3]:
BACKENDS = ("chroma", "pgvector")
BATCH_SIZE = 256
QUERY_REPEATS = 3
K = 5
MAX_CHUNKS = None
MAX_QUERIES = None

{
    "backends": BACKENDS,
    "batch_size": BATCH_SIZE,
    "query_repeats": QUERY_REPEATS,
    "k": K,
    "max_chunks": MAX_CHUNKS,
    "max_queries": MAX_QUERIES,
}

{'backends': ('chroma', 'pgvector'),
 'batch_size': 256,
 'query_repeats': 3,
 'k': 5,
 'max_chunks': None,
 'max_queries': None}

## Run Benchmark

This returns raw pandas DataFrames. It uses temporary benchmark collections and cleans them up after each run.

In [4]:
results = run_backend_benchmark(
    backends=BACKENDS,
    batch_size=BATCH_SIZE,
    query_repeats=QUERY_REPEATS,
    k=K,
    max_chunks=MAX_CHUNKS,
    max_queries=MAX_QUERIES,
    cleanup=True,
)

summaries = summarise_results(results)
results["parameters"]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6944.99it/s]


{'backends': ['chroma', 'pgvector'],
 'batch_size': 256,
 'query_repeats': 3,
 'k': 5,
 'max_chunks': None,
 'max_queries': None,
 'chunks_benchmarked': 5913,
 'reference_queries': 20,
 'collection_prefix': 'tpi_vectors_benchmark',
 'benchmark_id_prefix': 'tpi_vectors_benchmark__',
 'cleanup': True}

In [5]:
pd.DataFrame(results["errors"])

""


## Ingestion Speed

`total_embed_and_store_seconds` matches the professor's requested embed + store timing. `store_add_seconds` isolates just the backend write cost.

In [6]:
display(summaries.get("ingestion_summary", pd.DataFrame()))
display(results["ingestion_batches"])

,backend,chunks,batches,embed_seconds,store_add_seconds,total_embed_and_store_seconds,chunks_per_second_total,chunks_per_second_store_only
0,chroma,5913,24,32.678650,4.198599,36.877249,160.342763,1408.327005
1,pgvector,5913,24,28.219324,2.495041,30.714364,192.515786,2369.901175


,backend,batch_number,batch_size,store_add_seconds,store_chunks_per_second
0,chroma,1,256,0.180964,1414.648289
1,chroma,2,256,0.162379,1576.562992
2,chroma,3,256,0.161738,1582.804715
3,chroma,4,256,0.176848,1447.571818
4,chroma,5,256,0.178519,1434.017013
5,chroma,6,256,0.163281,1567.850491
6,chroma,7,256,0.162832,1572.171744
7,chroma,8,256,0.167365,1529.591388
8,chroma,9,256,0.160704,1592.994558
9,chroma,10,256,0.183529,1394.877774


## Query Latency

The professor asked for three runs and median latency. The summary table includes median, mean, p95, min, and max.

In [7]:
display(summaries.get("query_latency_summary", pd.DataFrame()))
display(results["query_latency"].head(30))

,backend,filter_name,runs,median_seconds,mean_seconds,p95_seconds,min_seconds,max_seconds
0,chroma,company_filter,60,0.005224,0.005189,0.005835,0.004525,0.006585
4,pgvector,company_filter,60,0.004896,0.005089,0.007792,0.002896,0.009749
1,chroma,sector_filter,60,0.004196,0.004259,0.005370,0.002762,0.005826
5,pgvector,sector_filter,60,0.007087,0.007283,0.009807,0.005149,0.010756
2,chroma,unfiltered,60,0.000982,0.001066,0.001257,0.000843,0.004044
6,pgvector,unfiltered,60,0.013581,0.013837,0.016287,0.011821,0.018594
3,chroma,year_filter,60,0.005237,0.005304,0.006198,0.004453,0.006705
7,pgvector,year_filter,60,0.007166,0.007296,0.009917,0.004456,0.010586


,backend,filter_name,repeat,query,k,seconds,returned,top_chunk_id,top_score
0,chroma,unfiltered,1,What are AGL's emissions targets?,5,0.004044,5,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0020,0.732535
1,chroma,unfiltered,1,What power stations does AGL operate?,5,0.001070,5,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0065,0.586245
2,chroma,unfiltered,1,What is AGL's approach to renewable energy investment?,5,0.000990,5,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0067,0.776811
3,chroma,unfiltered,1,How much electricity and natural gas did Alliant Energy's energy efficiency programs save in 2017?,5,0.000944,5,tpi_vectors_benchmark__Alliant_3-Alliant-Energy-Sustainability-Performance-Data-v08-02-2018_elem_0037,0.693020
4,chroma,unfiltered,1,What are Alliant Energy's Clean Energy Vision goals?,5,0.000915,5,tpi_vectors_benchmark__Alliant_CDP_2022_elem_0034,0.672022
5,chroma,unfiltered,1,How has Capital Power's transition toward a lower-emitting and renewable energy portfolio contributed to its strategic advantage?,5,0.001255,5,tpi_vectors_benchmark__Capital_Power_CDP2017_elem_0088,0.599546
6,chroma,unfiltered,1,What are CenterPoint Energy's Scope 1 and Scope 2 greenhouse gas emissions goals?,5,0.001247,5,tpi_vectors_benchmark__Center_Point_Energy_Transition_Goals_-_CenterPoint_Energy_Sustainability_elem_0001,0.799538
7,chroma,unfiltered,1,What are Chubu Electric Power Group's 2050 zero-emissions goals and supporting decarbonization actions?,5,0.001020,5,tpi_vectors_benchmark__Chubu_Zero_Emissions_Challenge_(March_2021)_elem_0000,0.852365
8,chroma,unfiltered,1,"What are BHP's Scope 1, Scope 2, and Scope 3 emissions reduction targets for 2030?",5,0.001054,5,tpi_vectors_benchmark__BHP_200910_BHPClimateChangeReport_2020_elem_0034,0.762394
9,chroma,unfiltered,1,What are Freeport-McMoRan's four 2030 Scope 1 and Scope 2 GHG emissions reduction targets?,5,0.001137,5,tpi_vectors_benchmark__Freeport_2022_Climate-Report_elem_0126,0.658937


## Recall@5, Precision@5, and MRR

In [8]:
display(summaries.get("retrieval_quality_summary", pd.DataFrame()))
display(results["retrieval_quality"])

,backend,mean_mrr,mean_recall@5,mean_precision@5
0,chroma,0.4125,0.426667,0.18
1,pgvector,0.4125,0.426667,0.18


,backend,query,recall@5,precision@5,mrr,relevant_available,relevant_returned
0,chroma,"How did Orkla define short, medium, and long-term time horizons for climate risks and targets in 2024?",0.000000,0.0,0.000000,1,0
1,chroma,How does Rio Tinto calculate Scope 3 emissions from processing copper concentrate?,0.000000,0.0,0.000000,2,0
2,chroma,"How does Rio Tinto define Scope 1, Scope 2, and Scope 3 emissions?",0.000000,0.0,0.000000,1,0
3,chroma,How has Capital Power's transition toward a lower-emitting and renewable energy portfolio contributed to its strategic advantage?,0.000000,0.0,0.000000,3,0
4,chroma,How much electricity and natural gas did Alliant Energy's energy efficiency programs save in 2017?,0.500000,0.2,0.333333,2,1
5,chroma,What are AGL's emissions targets?,0.250000,0.2,0.500000,4,1
6,chroma,What are Alliant Energy's Clean Energy Vision goals?,0.000000,0.0,0.000000,3,0
7,chroma,"What are BHP's Scope 1, Scope 2, and Scope 3 emissions reduction targets for 2030?",0.200000,0.2,0.333333,5,1
8,chroma,What are CenterPoint Energy's Scope 1 and Scope 2 greenhouse gas emissions goals?,0.666667,0.4,1.000000,3,2
9,chroma,What are Chubu Electric Power Group's 2050 zero-emissions goals and supporting decarbonization actions?,0.666667,0.4,1.000000,3,2


## Scores and Ranking Parity

This checks whether both backends return the same top-k chunk IDs in the same order.

In [9]:
display(summaries.get("score_summary", pd.DataFrame()))
display(summaries.get("ranking_parity_summary", pd.DataFrame()))
display(results["ranking_parity"])
display(results["topk_results"].head(30))

,backend,returned_scores,mean_score,median_score,min_score,max_score
0,chroma,100,0.674787,0.672781,0.537894,0.852365
1,pgvector,100,0.674787,0.672781,0.537894,0.852365


,baseline_backend,comparison_backend,queries_compared,same_order_queries,same_set_queries
0,chroma,pgvector,20,20,20


,baseline_backend,comparison_backend,query,same_order,same_set,baseline_ids,comparison_ids
0,chroma,pgvector,"How did Orkla define short, medium, and long-term time horizons for climate risks and targets in 2024?",True,True,"[tpi_vectors_benchmark__Orkla_CDP_2023_elem_0021, tpi_vectors_benchmark__Orkla_CDP_2023_elem_0081, tpi_vectors_benchmark__Orkla_CDP_2023...","[tpi_vectors_benchmark__Orkla_CDP_2023_elem_0021, tpi_vectors_benchmark__Orkla_CDP_2023_elem_0081, tpi_vectors_benchmark__Orkla_CDP_2023..."
1,chroma,pgvector,How does Rio Tinto calculate Scope 3 emissions from processing copper concentrate?,True,True,"[tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-123-report-addendum_elem_0070, tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-...","[tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-123-report-addendum_elem_0070, tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-..."
2,chroma,pgvector,"How does Rio Tinto define Scope 1, Scope 2, and Scope 3 emissions?",True,True,"[tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-123-report-addendum_elem_0037, tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-...","[tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-123-report-addendum_elem_0037, tpi_vectors_benchmark__Rio_Tinto_2024-climate-scope-..."
3,chroma,pgvector,How has Capital Power's transition toward a lower-emitting and renewable energy portfolio contributed to its strategic advantage?,True,True,"[tpi_vectors_benchmark__Capital_Power_CDP2017_elem_0088, tpi_vectors_benchmark__Capital_Power_CDP2017_elem_0034, tpi_vectors_benchmark__...","[tpi_vectors_benchmark__Capital_Power_CDP2017_elem_0088, tpi_vectors_benchmark__Capital_Power_CDP2017_elem_0034, tpi_vectors_benchmark__..."
4,chroma,pgvector,How much electricity and natural gas did Alliant Energy's energy efficiency programs save in 2017?,True,True,"[tpi_vectors_benchmark__Alliant_3-Alliant-Energy-Sustainability-Performance-Data-v08-02-2018_elem_0037, tpi_vectors_benchmark__Alliant_C...","[tpi_vectors_benchmark__Alliant_3-Alliant-Energy-Sustainability-Performance-Data-v08-02-2018_elem_0037, tpi_vectors_benchmark__Alliant_C..."
5,chroma,pgvector,What are AGL's emissions targets?,True,True,"[tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0020, tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0041, tpi_vec...","[tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0020, tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0041, tpi_vec..."
6,chroma,pgvector,What are Alliant Energy's Clean Energy Vision goals?,True,True,"[tpi_vectors_benchmark__Alliant_CDP_2022_elem_0034, tpi_vectors_benchmark__Alliant_CDP_2022_elem_0234, tpi_vectors_benchmark__Alliant_CD...","[tpi_vectors_benchmark__Alliant_CDP_2022_elem_0034, tpi_vectors_benchmark__Alliant_CDP_2022_elem_0234, tpi_vectors_benchmark__Alliant_CD..."
7,chroma,pgvector,"What are BHP's Scope 1, Scope 2, and Scope 3 emissions reduction targets for 2030?",True,True,"[tpi_vectors_benchmark__BHP_200910_BHPClimateChangeReport_2020_elem_0034, tpi_vectors_benchmark__Freeport_2022_Climate-Report_elem_0126,...","[tpi_vectors_benchmark__BHP_200910_BHPClimateChangeReport_2020_elem_0034, tpi_vectors_benchmark__Freeport_2022_Climate-Report_elem_0126,..."
8,chroma,pgvector,What are CenterPoint Energy's Scope 1 and Scope 2 greenhouse gas emissions goals?,True,True,"[tpi_vectors_benchmark__Center_Point_Energy_Transition_Goals_-_CenterPoint_Energy_Sustainability_elem_0001, tpi_vectors_benchmark__Cente...","[tpi_vectors_benchmark__Center_Point_Energy_Transition_Goals_-_CenterPoint_Energy_Sustainability_elem_0001, tpi_vectors_benchmark__Cente..."
9,chroma,pgvector,What are Chubu Electric Power Group's 2050 zero-emissions goals and supporting decarbonization actions?,True,True,"[tpi_vectors_benchmark__Chubu_Zero_Emissions_Challenge_(March_2021)_elem_0000, tpi_vectors_benchmark__Chubu_Zero_Emissions_Challenge_(Ma...","[tpi_vectors_benchmark__Chubu_Zero_Emissions_Challenge_(March_2021)_elem_0000, tpi_vec

,backend,query,rank,chunk_id,score,is_relevant,text_preview
0,chroma,What are AGL's emissions targets?,1,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0020,0.732535,False,RUNNING TITLE: affordable and sustainable energy options;\n\nHEADER (H2): 4\n\nMuch of the focus on AGL’s emissions is concentrated on t...
1,chroma,What are AGL's emissions targets?,2,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0041,0.713536,True,RUNNING TITLE: Source: Department of Environment (2014)\n\nHEADER (H2): 8\n\nTable 2 outlines the key parameters for establishing a Nati...
2,chroma,What are AGL's emissions targets?,3,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0050,0.713106,False,"RUNNING TITLE: Scenario 2 – '2 degree'\n\nHEADER (H2): 10\n\nemissions constraints, including: a carbon tax; emissions trading; regulati..."
3,chroma,What are AGL's emissions targets?,4,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0016,0.687635,False,RUNNING TITLE: CARBON CONSTRAINED FUTURE AGL’s approach to climate change mitigation: a scenario analysis\n\nHEADER (H2): INTRODUCTION\n...
4,chroma,What are AGL's emissions targets?,5,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0017,0.686768,False,RUNNING TITLE: affordable and sustainable energy options;\n\nHEADER (H2): 4\n\nMEASUREMENT OF GREENHOUSE GAS EMISSIONS AGL uses three ap...
5,chroma,What power stations does AGL operate?,1,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0065,0.586245,False,RUNNING TITLE: Figure 5: NPV Analysis of Emission Reduction Scenarios\n\nHEADER (H2): 12\n\nAs one of Australia’s largest electricity re...
6,chroma,What power stations does AGL operate?,2,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0019,0.583468,True,RUNNING TITLE: affordable and sustainable energy options;\n\nHEADER (H2): 4\n\nPREPARING FOR THE DECOMMISSIONING OF AGL’S EXISTING GENER...
7,chroma,What power stations does AGL operate?,3,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0058,0.582571,False,"RUNNING TITLE: Base NPV\n\nHEADER (H2): 11\n\nHowever, it is worth noting that for carbon pricing to be utilised effectively to drive em..."
8,chroma,What power stations does AGL operate?,4,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0006,0.553008,True,RUNNING TITLE: CARBON CONSTRAINED FUTURE AGL’s approach to climate change mitigation: a scenario analysis\n\nHEADER (H2): INTRODUCTION\n...
9,chroma,What power stations does AGL operate?,5,tpi_vectors_benchmark__AGL_Carbon_Constrained_Future_elem_0028,0.537894,False,"RUNNING TITLE: Bayswater\n\nHEADER (H2): 6\n\nSunverge: In February 2016, AGL invested $20 million in Sunverge which is a US-based energ..."


## Metadata Filters

The professor asks for company, year, and sector filtering. This table shows which `where` filters are available from the current chunk metadata.

In [10]:
display(results["metadata_filters"])
display(results["query_latency"].groupby(["backend", "filter_name"], as_index=False)["seconds"].median())

,filter_name,where,available
0,unfiltered,None,True
1,company_filter,{'company': 'AGL'},True
2,year_filter,{'year': '2016'},True
3,sector_filter,{'sector': 'Energy Utilities'},True


,backend,filter_name,seconds
0,chroma,company_filter,0.005224
1,chroma,sector_filter,0.004196
2,chroma,unfiltered,0.000982
3,chroma,year_filter,0.005237
4,pgvector,company_filter,0.004896
5,pgvector,sector_filter,0.007087
6,pgvector,unfiltered,0.013581
7,pgvector,year_filter,0.007166


## Deployment Complexity

Concrete counts for setup complexity: extra services, Docker Compose line count, and clean-machine backend setup steps.

In [11]:
display(summaries.get("deployment_complexity_summary", pd.DataFrame()))

,backend,extra_services_required,docker_compose_total_lines,docker_compose_noncomment_lines,clean_machine_backend_steps,setup_notes
0,chroma,0,0,0,2,Install Python env; set VECTOR_STORE=chroma.
1,pgvector,1,26,23,5,Install Docker; set Postgres env vars; docker compose up; run Alembic; run pgvector smoke test.


## Code Legibility

Concrete counts for each backend implementation. If `radon` is installed in the kernel, this also includes cognitive-complexity scores.

In [12]:
display(summaries.get("code_legibility_summary", pd.DataFrame()))

,backend,path,line_count,import_count,radon_functions_classes,radon_max_complexity,radon_mean_complexity,radon_note
0,chroma,vector_store/chroma.py,110,5,7,4,2.142857,
1,pgvector,vector_store/pgvector.py,174,11,10,6,2.500000,


## Optional Save

Only run this after the tables look right.

In [13]:
# write_results_json(results, "evaluation_results.json")